# Modelado supervisado — Predicción de Popularidad de Canciones en Spotify
### Evaluación Parcial N°2 — MLY1101 Machine Learning

Este notebook cubre las etapas de **Modelado** y **Evaluación** de CRISP-DM. Como `popularity` es una variable numérica (0-100), el problema es de **regresión**. Se comparan dos modelos supervisados contra una línea base:

| Modelo | Rol | Justificación |
|---|---|---|
| Baseline (media) | Referencia mínima | Si un modelo no supera predecir siempre el promedio, no aporta valor. |
| Regresión lineal | Modelo base interpretable | Simple y rápido; permite saber cuánto se explica con relaciones lineales. |
| Gradient Boosting | Modelo más potente | Captura relaciones no lineales e interacciones (p. ej. género × audio features), que el EDA sugería porque las correlaciones lineales eran débiles. |

Usa el dataset de `01_preparacion_datos.ipynb` (una fila por canción, géneros en multi-hot).

## 5. Modelado

### 5.1 Carga de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style('whitegrid')
RANDOM_STATE = 42

df = pd.read_csv('../data/processed/spotify_tracks_clean.csv')
genre_cols = [c for c in df.columns if c.startswith('genre_')]
print(df.shape)

### 5.2 Variables predictoras y partición entrenamiento / prueba

- **Predictores (128):** 13 audio features, `explicit` y las 114 columnas `genre_*`.
- **Se excluyen** `track_id`, `track_name`, `artists` y `album_name`: son identificadores/texto, no características de la canción, y usarlos haría que el modelo memorice en vez de generalizar.
- **Partición 80/20** con semilla fija. Como hay una sola fila por canción, **la misma canción no puede quedar en entrenamiento y en prueba a la vez** (se evita la fuga de datos que había antes de corregir los duplicados).

In [ ]:
audio_cols = ['duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
              'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']

X = df[audio_cols + ['explicit'] + genre_cols].copy()
X['explicit'] = X['explicit'].astype(int)
y = df['popularity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print('Predictores:', X.shape[1], '| train:', X_train.shape[0], '| test:', X_test.shape[0])

### 5.3 Línea base y regresión lineal

Métricas usadas (en unidades de `popularity`, escala 0-100):
- **RMSE:** error típico, penaliza más los errores grandes.
- **MAE:** error absoluto promedio, fácil de explicar al negocio.
- **R²:** proporción de la variabilidad de `popularity` que explica el modelo (0 = nada, 1 = todo).

In [ ]:
def metricas(y_real, y_pred):
    return {'RMSE': mean_squared_error(y_real, y_pred) ** 0.5,
            'MAE': mean_absolute_error(y_real, y_pred),
            'R2': r2_score(y_real, y_pred)}

resultados = {}
pred_base = np.full(len(y_test), y_train.mean())
resultados['Baseline (media)'] = metricas(y_test, pred_base)
resultados['Baseline (media)']

In [ ]:
lin = LinearRegression().fit(X_train, y_train)
pred_lin = lin.predict(X_test)
resultados['Regresión lineal'] = metricas(y_test, pred_lin)
resultados['Regresión lineal']

### 5.4 Gradient Boosting

Se usa `HistGradientBoostingRegressor` de scikit-learn, una implementación de gradient boosting sobre árboles que entrena mucho más rápido en datasets grandes (≈ 90.000 filas). Los hiperparámetros se eligen con **búsqueda aleatoria y validación cruzada de 3 particiones sobre el conjunto de entrenamiento**, así el conjunto de prueba solo se usa para la evaluación final. (La búsqueda tarda unos 3 minutos.)

- `learning_rate`, `max_iter`: velocidad de aprendizaje y número de árboles.
- `max_depth`, `min_samples_leaf`: complejidad de cada árbol (controlan el sobreajuste).
- `l2_regularization`: penalización para suavizar el modelo.

In [ ]:
param_dist = {
    'learning_rate': [0.03, 0.05, 0.1],
    'max_iter': [200, 400],
    'max_depth': [None, 6, 10],
    'min_samples_leaf': [20, 50, 100],
    'l2_regularization': [0, 1, 10],
}
busqueda = RandomizedSearchCV(
    HistGradientBoostingRegressor(random_state=RANDOM_STATE),
    param_distributions=param_dist, n_iter=8, cv=3,
    scoring='neg_root_mean_squared_error', random_state=RANDOM_STATE, n_jobs=-1)
busqueda.fit(X_train, y_train)

gb = busqueda.best_estimator_
print('Mejores hiperparámetros:', busqueda.best_params_)
print('RMSE en validación cruzada (train):', round(-busqueda.best_score_, 3))

pred_gb = gb.predict(X_test)
resultados['Gradient Boosting'] = metricas(y_test, pred_gb)
resultados['Gradient Boosting']

## 6. Evaluación

### 6.1 Comparación de modelos y gráfico real vs predicho

In [ ]:
tabla = pd.DataFrame(resultados).T.round(3)
display(tabla) if 'display' in globals() else print(tabla)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, pred, nombre in zip(axes, [pred_lin, pred_gb], ['Regresión lineal', 'Gradient Boosting']):
    ax.scatter(y_test, pred, alpha=0.08, s=6)
    ax.plot([0, 100], [0, 100], 'r--', label='Predicción perfecta')
    ax.set_xlabel('Popularity real'); ax.set_ylabel('Popularity predicha')
    ax.set_title(f'{nombre} (R² = {r2_score(y_test, pred):.2f})')
    ax.legend()
plt.tight_layout()
plt.savefig('../images/real_vs_predicho.png', dpi=100)
plt.show()

**Lectura de los resultados (conjunto de prueba):**

| Modelo | RMSE | MAE | R² |
|---|---|---|---|
| Baseline (media) | 20,45 | 17,12 | 0,00 |
| Regresión lineal | 16,50 | 11,81 | 0,35 |
| Gradient Boosting | **15,11** | **10,53** | **0,45** |

- Ambos modelos superan claramente la línea base: el género y las audio features sí contienen señal sobre la popularidad.
- **Gradient Boosting mejora a la regresión lineal** en todas las métricas (RMSE -1,4 puntos, MAE -1,3 puntos, R² de 0,35 a 0,45). Eso confirma que hay relaciones no lineales y que un modelo más flexible las aprovecha.
- Aun así **R² ≈ 0,45**: más de la mitad de la variabilidad de `popularity` no se explica. Es coherente con la hipótesis del EDA: faltan factores externos (artista, promoción, fecha de lanzamiento) que no están en el dataset.
- En promedio el mejor modelo se equivoca por ≈ 10,5 puntos de popularidad (MAE), sobre una escala 0-100.
- En el gráfico, si el modelo fuera perfecto los puntos estarían sobre la línea roja. Las predicciones se concentran en la zona media y las canciones con popularidad real muy alta o igual a 0 se predicen peor; Gradient Boosting se acerca más a la diagonal que la regresión lineal.

### 6.2 Evaluación del KPI de negocio: canciones "top" (popularity > 70)

El segundo KPI definido en la EP1 era identificar las canciones top. Solo ≈ 3% de las canciones lo son, así que se mide con **recall** (de las top reales, cuántas detecta) y **precisión** (de las que marca como top, cuántas lo son), según el umbral de predicción, y con un enfoque más realista para una distribuidora: **priorizar el 10% de canciones con mayor predicción**.

In [ ]:
real_top = (y_test > 70)
print('Canciones top en test:', real_top.sum(), f'({real_top.mean():.1%})\n')

filas = []
for nombre, pred in [('Regresión lineal', pred_lin), ('Gradient Boosting', pred_gb)]:
    for umbral in [70, 60, 50]:
        pred_top = pred > umbral
        tp = (pred_top & real_top).sum()
        filas.append({'Modelo': nombre, 'Umbral predicho >': umbral,
                      'Canciones marcadas': pred_top.sum(),
                      'Recall (top detectadas)': tp / real_top.sum(),
                      'Precisión': tp / max(pred_top.sum(), 1)})
kpi = pd.DataFrame(filas).round(3)
print(kpi.to_string(index=False))

# Priorizar el 10% de canciones con mayor popularidad predicha
n10 = int(len(y_test) * 0.10)
for nombre, pred in [('Regresión lineal', pred_lin), ('Gradient Boosting', pred_gb)]:
    idx = np.argsort(pred)[::-1][:n10]
    print(f'{nombre}: al priorizar el 10% con mayor predicción se captura {real_top.values[idx].sum() / real_top.sum():.1%} de las canciones top')

**Interpretación para el negocio:**
- Con el umbral estricto (predicho > 70) Gradient Boosting casi no marca canciones (22 de ~18.000): cuando lo hace acierta ~45% de las veces, pero **detecta solo ~2% de las top**. Los modelos de regresión tienden a predecir hacia el promedio y rara vez llegan a valores extremos.
- Como herramienta de **priorización**, el modelo es más útil: si la distribuidora promociona solo el 10% de canciones con mayor predicción, **captura ~32% de las canciones top** (la regresión lineal captura ~23%; elegir al azar capturaría ~10%). Es decir, concentra ~3 veces más canciones exitosas que el azar.
- Conclusión: el modelo sirve para **ordenar y priorizar candidatas**, no para asegurar que una canción será un éxito.

### 6.3 Importancia de variables

Se usa **importancia por permutación**: cuánto empeora el RMSE al desordenar una variable (mayor valor = más importante).

In [ ]:
muestra = X_test.sample(5000, random_state=RANDOM_STATE)
imp = permutation_importance(gb, muestra, y_test.loc[muestra.index], n_repeats=3,
                             random_state=RANDOM_STATE, scoring='neg_root_mean_squared_error', n_jobs=-1)
importancia = pd.Series(imp.importances_mean, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
importancia.head(15).sort_values().plot(kind='barh', color='steelblue')
plt.title('Importancia de variables (Gradient Boosting)')
plt.xlabel('Aumento del RMSE al desordenar la variable')
plt.tight_layout()
plt.savefig('../images/importancia_variables.png', dpi=100)
plt.show()
print(importancia.head(15).round(3))

**Interpretación:**
- Entre las variables más importantes aparecen géneros muy distintos en popularidad promedio: `iranian`, `latin` y `romance` (muy baja) y `k-pop` (alta). Saber el género dice mucho sobre qué tan popular será una canción.
- Entre las audio features destacan `acousticness`, `loudness`, `instrumentalness`, `energy` y `duration_ms`.
- **Recomendación:** la popularidad depende mucho del contexto de género; conviene evaluar las canciones **dentro de su género** y no solo comparando todo el catálogo. Importancia no implica causalidad.

### 6.4 Guardado de modelos

In [ ]:
joblib.dump(lin, '../models/regresion_lineal.joblib')
joblib.dump(gb, '../models/gradient_boosting.joblib')
joblib.dump(list(X.columns), '../models/columnas_modelo.joblib')
tabla.to_csv('../models/metricas_modelos.csv')

## 7. Conclusiones de la etapa de modelado

1. **Gradient Boosting es el mejor modelo** (RMSE 15,11, MAE 10,53, R² 0,45) y supera a la regresión lineal y a la línea base.
2. El rendimiento tiene un **techo** por la falta de variables como artista, promoción o fecha de lanzamiento.
3. Para la distribuidora, el modelo es una **herramienta de priorización** (captura ~32% de las canciones top en el 10% mejor rankeado), no un predictor exacto, y debe complementar el criterio de A&R.
4. Siguiente etapa: **aprendizaje no supervisado** (clustering) para descubrir segmentos de canciones y relacionarlos con la popularidad (`04_clustering.ipynb`).